# 🔧 Sesión 09-B: Fine-Tuning — Estrategias Avanzadas de Transfer Learning

**Curso:** IS056B — Ingeniería del Conocimiento
**Docente:** Msc. Jaime Antonio Huaytalla Pariona
**Universidad:** UNCP — Facultad de Ingeniería de Sistemas
**Semestre:** 2026-II | **Unidad II:** Transferencia de Aprendizaje, Series de Tiempo y MLOps

---

### 🎯 Objetivos de este notebook
1. **Dominar** el protocolo de 2 fases para Fine-Tuning seguro
2. **Experimentar** con diferentes profundidades de descongelamiento
3. **Comparar** las 3 estrategias: CNN desde cero vs. Feature Extraction vs. Fine-Tuning
4. **Exportar** el modelo final a TFLite para despliegue móvil

### 📚 Estructura
| Ejemplo | Nivel | Dataset | Enfoque |
|---------|-------|---------|---------|
| 1 | 🟢 Básico | Cats vs Dogs | Primer Fine-Tuning con protocolo de 2 fases |
| 2 | 🟡 Intermedio | TF Flowers | Profundidad de descongelamiento + BN demo |
| 3 | 🔴 Avanzado | Oxford-IIIT Pet (37 razas) | Comparación 3 estrategias + Mixup + TFLite |

### ⚠️ Prerrequisito
Este notebook asume que ya completaste la **Sesión 09-A** (Feature Extraction). Los conceptos de backbone, `include_top=False`, `trainable=False` y `preprocess_input` se dan por conocidos.

## ⚙️ Configuración del Entorno

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True)
print("GPU:", gpu_info.stdout.strip() or "No GPU -- activa T4 en Runtime > Change runtime type")

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

---
# 📖 ¿Qué es Fine-Tuning?

## Feature Extraction vs Fine-Tuning

| | Feature Extraction (S09-A) | Fine-Tuning (hoy) |
|---|---|---|
| **Backbone** | 🔒 Congelado completamente | 🔓 Últimas N capas descongeladas |
| **Qué se entrena** | Solo el clasificador nuevo | Clasificador + capas profundas |
| **Learning rate** | Normal (1e-3) | Muy bajo (1e-5) |
| **Riesgo** | Bajo | Medio (puede destruir features) |
| **Accuracy** | Bueno | Mejor |

## El protocolo obligatorio de 2 fases

```
FASE 1: Feature Extraction (warm-up)
├── Congelar todo el backbone
├── Entrenar solo el clasificador (5-10 épocas)
└── Propósito: estabilizar el clasificador antes de tocar el backbone

FASE 2: Fine-Tuning
├── Descongelar las últimas N capas del backbone
├── Recompilar con LR 10x-100x menor
└── Entrenar todo junto (10-30 épocas más)
```

> ⚠️ **¿Por qué 2 fases?** Si descongelamos desde el inicio, el clasificador (pesos aleatorios) produce gradientes enormes y ruidosos que **destruyen** los features pre-entrenados del backbone. La Fase 1 estabiliza el clasificador para que sus gradientes sean "suaves" en la Fase 2.

---
# 🟢 Ejemplo 1 — Básico: Fine-Tuning en Cats vs Dogs

**Objetivo:** Primer Fine-Tuning paso a paso con el protocolo de 2 fases.

**Dataset:** Cats vs Dogs (~25,000 imágenes, clasificación binaria).

**Modelo:** MobileNetV2 (ligero, ideal para practicar).

### 1.1. Cargar y preprocesar datos

In [ ]:
# Cargar Cats vs Dogs
(train_raw_1, val_raw_1), info_1 = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True, with_info=True
)

NUM_CLASSES_1 = 2
CLASS_NAMES_1 = ['cat', 'dog']

def preprocess_1(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        tf.cast(image, tf.float32))
    label = tf.one_hot(label, NUM_CLASSES_1)
    return image, label

train_ds_1 = (train_raw_1.map(preprocess_1, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_ds_1 = (val_raw_1.map(preprocess_1, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print(f"Clases: {CLASS_NAMES_1}")
print(f"Train batches: {tf.data.experimental.cardinality(train_ds_1).numpy()}")
print(f"Val batches:   {tf.data.experimental.cardinality(val_ds_1).numpy()}")

### 1.2. FASE 1 — Feature Extraction (warm-up del clasificador)

Primero entrenamos **solo** el clasificador nuevo con el backbone congelado.

In [ ]:
# Cargar backbone
base_model_1 = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model_1.trainable = False  # CONGELAR TODO

# Construir modelo
inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = base_model_1(inputs, training=False)  # BN en modo inferencia
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES_1, activation='softmax')(x)

model_1 = models.Model(inputs, outputs, name='catdog_mobilenetv2_FT')

# Compilar Fase 1 con LR normal
model_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_p1 = sum(tf.size(w).numpy() for w in model_1.trainable_weights)
print(f"=== FASE 1: Feature Extraction (warm-up) ===")
print(f"Params entrenables: {trainable_p1:,} (solo clasificador)")
print(f"Backbone congelado: {base_model_1.count_params():,} params")

In [ ]:
# Entrenar Fase 1
history_p1 = model_1.fit(
    train_ds_1, validation_data=val_ds_1,
    epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)

fe_acc = max(history_p1.history['val_accuracy'])
print(f"\nFase 1 completada — Best val_accuracy: {fe_acc:.4f}")
print(f"Este es nuestro BASELINE de Feature Extraction")

### 1.3. FASE 2 — Fine-Tuning: Descongelar y re-entrenar

Ahora descongelamos las **últimas 30 capas** de MobileNetV2 y recompilamos con un **learning rate 100x menor**.

In [ ]:
# === DESCONGELAR las últimas 30 capas ===
base_model_1.trainable = True

total_layers = len(base_model_1.layers)
FINE_TUNE_FROM = total_layers - 30  # Congelar todas excepto las últimas 30

for layer in base_model_1.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

# Verificar
frozen = sum(1 for l in base_model_1.layers if not l.trainable)
unfrozen = sum(1 for l in base_model_1.layers if l.trainable)
trainable_p2 = sum(tf.size(w).numpy() for w in model_1.trainable_weights)

print(f"=== FASE 2: Fine-Tuning ===")
print(f"Total capas backbone: {total_layers}")
print(f"Capas congeladas: {frozen} | Capas descongeladas: {unfrozen}")
print(f"Params entrenables ahora: {trainable_p2:,} (vs {trainable_p1:,} en Fase 1)")
print(f"Factor de aumento: {trainable_p2/trainable_p1:.1f}x más parámetros")

In [ ]:
# RECOMPILAR con LR mucho más bajo (obligatorio después de cambiar trainable)
model_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # 100x menor que Fase 1
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenar Fase 2 (continuar desde donde Fase 1 terminó)
history_p2 = model_1.fit(
    train_ds_1, validation_data=val_ds_1,
    epochs=15,
    initial_epoch=history_p1.epoch[-1] + 1,  # Continuar numeración de épocas
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    ]
)

ft_acc = max(history_p2.history['val_accuracy'])
print(f"\nFase 2 completada — Best val_accuracy: {ft_acc:.4f}")
print(f"Mejora sobre Feature Extraction: {(ft_acc - fe_acc)*100:+.2f} puntos porcentuales")

### 1.4. Visualizar las 2 fases

In [ ]:
# Combinar historiales
all_acc = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
all_loss = history_p1.history['val_loss'] + history_p2.history['val_loss']
all_train_acc = history_p1.history['accuracy'] + history_p2.history['accuracy']
all_train_loss = history_p1.history['loss'] + history_p2.history['loss']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Línea divisoria entre fases
phase1_end = len(history_p1.history['accuracy'])
for ax in axes:
    ax.axvline(x=phase1_end + 0.5, color='gray', linestyle='--', alpha=0.7, linewidth=2)

epochs_all = range(1, len(all_acc) + 1)

# Accuracy
axes[0].plot(epochs_all, all_train_acc, 'o-', label='Train', color='#2196F3', alpha=0.7)
axes[0].plot(epochs_all, all_acc, 's-', label='Val', color='#FF5722', linewidth=2)
axes[0].fill_betweenx([min(all_acc)-0.01, max(all_acc)+0.01],
                      0.5, phase1_end+0.5, alpha=0.08, color='blue')
axes[0].fill_betweenx([min(all_acc)-0.01, max(all_acc)+0.01],
                      phase1_end+0.5, len(all_acc)+0.5, alpha=0.08, color='green')
axes[0].text(phase1_end/2, min(all_acc), 'FASE 1\nFeature Extraction',
            ha='center', fontsize=10, color='blue', alpha=0.6, fontweight='bold')
axes[0].text(phase1_end + (len(all_acc)-phase1_end)/2, min(all_acc), 'FASE 2\nFine-Tuning',
            ha='center', fontsize=10, color='green', alpha=0.6, fontweight='bold')
axes[0].set_title('Accuracy — Protocolo de 2 Fases', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(epochs_all, all_train_loss, 'o-', label='Train', color='#2196F3', alpha=0.7)
axes[1].plot(epochs_all, all_loss, 's-', label='Val', color='#FF5722', linewidth=2)
axes[1].axvline(x=phase1_end + 0.5, color='gray', linestyle='--', alpha=0.7, linewidth=2)
axes[1].set_title('Loss — Protocolo de 2 Fases', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Fine-Tuning: MobileNetV2 en Cats vs Dogs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nResumen:")
print(f"  Feature Extraction (Fase 1): {fe_acc:.4f}")
print(f"  Fine-Tuning (Fase 2):        {ft_acc:.4f}")
print(f"  Mejora:                       {(ft_acc-fe_acc)*100:+.2f}pp")

### ✅ Conclusión Ejemplo 1

| Fase | Val Accuracy | LR | Params entrenables |
|------|-------------|-----|-------------------|
| **Fase 1** (FE) | ~97% | 1e-3 | ~17K |
| **Fase 2** (FT) | ~98.5% | 1e-5 | ~600K+ |

> El protocolo de 2 fases es clave: la Fase 1 estabiliza el clasificador antes de que la Fase 2 modifique el backbone. El resultado es un modelo que supera a Feature Extraction puro.

---
# 🟡 Ejemplo 2 — Intermedio: Profundidad de Descongelamiento en TF Flowers

**Objetivo:** Experimentar con diferentes profundidades de descongelamiento y demostrar el efecto de `training=False` en BatchNormalization.

**Dataset:** TF Flowers (3,670 imágenes, 5 clases).

**Modelo:** EfficientNetB0 — exploraremos cuántas capas descongelar.

### 2.1. Cargar y preparar datos

In [ ]:
(train_raw_2, val_raw_2), info_2 = tfds.load(
    'tf_flowers', split=['train[:80%]', 'train[80%:]'],
    as_supervised=True, with_info=True
)

NUM_CLASSES_2 = 5
CLASS_NAMES_2 = info_2.features['label'].names

# Augmentation
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

def prep_train_2(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = augmentation(image, training=True)
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    return image, tf.one_hot(label, NUM_CLASSES_2)

def prep_val_2(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.efficientnet.preprocess_input(tf.cast(image, tf.float32))
    return image, tf.one_hot(label, NUM_CLASSES_2)

train_ds_2 = (train_raw_2.map(prep_train_2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_ds_2 = (val_raw_2.map(prep_val_2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print(f"Clases: {CLASS_NAMES_2}")
print(f"Train: {tf.data.experimental.cardinality(train_ds_2).numpy()} batches")

### 2.2. Fase 1: Feature Extraction (baseline)

In [ ]:
def build_efficientnet_model(num_classes):
    base = tf.keras.applications.EfficientNetB0(
        weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = False

    inp = layers.Input(shape=(*IMG_SIZE, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inp, out), base

model_2, base_2 = build_efficientnet_model(NUM_CLASSES_2)
model_2.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='categorical_crossentropy', metrics=['accuracy'])

history_fe = model_2.fit(train_ds_2, validation_data=val_ds_2, epochs=10, verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                                restore_best_weights=True)])

fe_baseline = max(history_fe.history['val_accuracy'])
print(f"\nBaseline Feature Extraction: {fe_baseline:.4f}")

### 2.3. Explorar la estructura de EfficientNetB0

Antes de descongelar, necesitamos entender cómo está organizado el backbone.

In [ ]:
# Listar los bloques del modelo
print(f"Total capas en EfficientNetB0: {len(base_2.layers)}")
print(f"\nBloques principales:")
print("-" * 60)

blocks = {}
for i, layer in enumerate(base_2.layers):
    name = layer.name
    # Identificar el bloque
    for prefix in ['block1', 'block2', 'block3', 'block4', 'block5', 'block6', 'block7', 'stem', 'top']:
        if name.startswith(prefix):
            if prefix not in blocks:
                blocks[prefix] = {'start': i, 'end': i, 'count': 0}
            blocks[prefix]['end'] = i
            blocks[prefix]['count'] += 1
            break

for block_name, info in sorted(blocks.items()):
    params = sum(l.count_params() for l in base_2.layers[info['start']:info['end']+1])
    print(f"  {block_name:10s} | capas {info['start']:3d}–{info['end']:3d} "
          f"({info['count']:3d} capas) | {params:>10,} params")

### 2.4. Experimento: ¿Cuántas capas descongelar?

Probamos 4 profundidades de descongelamiento y comparamos los resultados.

In [ ]:
# Experimento de profundidad de descongelamiento
depths = [0, 20, 50, 100, len(base_2.layers)]  # 0 = FE puro, len = todo descongelado
depth_results = []

for depth in depths:
    print(f"\n{'='*50}")
    print(f"Probando: descongelar ultimas {depth} capas")
    print('='*50)

    # Reconstruir modelo fresco
    model_exp, base_exp = build_efficientnet_model(NUM_CLASSES_2)

    # Fase 1: FE warm-up
    model_exp.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                     loss='categorical_crossentropy', metrics=['accuracy'])
    h1 = model_exp.fit(train_ds_2, validation_data=val_ds_2, epochs=5, verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                                    restore_best_weights=True)])

    if depth > 0:
        # Fase 2: Descongelar
        base_exp.trainable = True
        freeze_from = max(0, len(base_exp.layers) - depth)
        for layer in base_exp.layers[:freeze_from]:
            layer.trainable = False

        model_exp.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                         loss='categorical_crossentropy', metrics=['accuracy'])
        h2 = model_exp.fit(train_ds_2, validation_data=val_ds_2, epochs=15, verbose=0,
            initial_epoch=h1.epoch[-1]+1,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                                        restore_best_weights=True)])
        best_acc = max(h2.history['val_accuracy'])
        total_epochs = len(h1.history['loss']) + len(h2.history['loss'])
    else:
        best_acc = max(h1.history['val_accuracy'])
        total_epochs = len(h1.history['loss'])

    trainable_count = sum(tf.size(w).numpy() for w in model_exp.trainable_weights)

    result = {
        'Capas descongeladas': depth,
        'Params entrenables': trainable_count,
        'Best Val Acc': best_acc,
        'Epocas totales': total_epochs,
    }
    depth_results.append(result)
    print(f"   val_accuracy = {best_acc:.4f} | params = {trainable_count:,}")

print("\nExperimento completado!")

In [ ]:
# Tabla de resultados
df_depth = pd.DataFrame(depth_results)
df_depth['Params entrenables'] = df_depth['Params entrenables'].apply(lambda x: f"{x:,}")
print("\nResultados del Experimento de Profundidad")
print("=" * 70)
print(df_depth.to_string(index=False))

In [ ]:
# Grafico del experimento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_labels = [str(d) for d in depths]
accs = [r['Best Val Acc'] for r in depth_results]
params = [int(r['Params entrenables'].replace(',', '')) if isinstance(r['Params entrenables'], str)
          else r['Params entrenables'] for r in depth_results]

# Accuracy vs profundidad
colors_depth = ['#E0E0E0'] + plt.cm.Greens(np.linspace(0.3, 0.9, len(depths)-1)).tolist()
bars1 = axes[0].bar(x_labels, accs, color=colors_depth, edgecolor='gray')
axes[0].set_title('Val Accuracy vs Capas Descongeladas', fontweight='bold')
axes[0].set_xlabel('Capas descongeladas')
axes[0].set_ylabel('Val Accuracy')
axes[0].set_ylim([min(accs) - 0.03, max(accs) + 0.02])
for bar, acc in zip(bars1, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{acc:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Params vs accuracy (trade-off)
axes[1].scatter([int(r['Params entrenables'].replace(',', '')) if isinstance(r['Params entrenables'], str)
                 else r['Params entrenables'] for r in depth_results],
               accs, c=range(len(depths)), cmap='RdYlGn', s=150, edgecolors='black', zorder=5)
for i, r in enumerate(depth_results):
    p = int(r['Params entrenables'].replace(',', '')) if isinstance(r['Params entrenables'], str) else r['Params entrenables']
    axes[1].annotate(f"{r['Capas descongeladas']} capas",
                    (p, r['Best Val Acc']),
                    textcoords="offset points", xytext=(10, 5), fontsize=9)
axes[1].set_title('Trade-off: Params Entrenables vs Accuracy', fontweight='bold')
axes[1].set_xlabel('Params entrenables')
axes[1].set_ylabel('Val Accuracy')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Experimento: Profundidad de Descongelamiento (EfficientNetB0)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5. Demo: El error silencioso de BatchNormalization

`training=False` vs `training=True` en el backbone durante Fine-Tuning. Este es el **error más común** en Transfer Learning.

In [ ]:
# Demo: training=True vs training=False en BN
# Construir DOS modelos identicos pero con BN diferente

def build_bn_demo_model(training_flag, label):
    base = tf.keras.applications.EfficientNetB0(
        weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))

    # Descongelar ultimas 20 capas
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False

    inp = layers.Input(shape=(*IMG_SIZE, 3))
    x = base(inp, training=training_flag)  # <-- LA DIFERENCIA
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES_2, activation='softmax')(x)
    model = models.Model(inp, out, name=f'bn_{label}')
    return model

# Modelo CORRECTO: BN en modo inferencia
model_bn_correct = build_bn_demo_model(training_flag=False, label='correct')
model_bn_correct.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                        loss='categorical_crossentropy', metrics=['accuracy'])

# Fase 1 (warm-up) para el modelo correcto
# Primero congelamos, entrenamos fase 1, luego descongelamos
# (simplificado: usamos solo fase 2 con pesos ya inicializados)

print("Entrenando modelo CORRECTO (training=False en BN)...")
h_correct = model_bn_correct.fit(train_ds_2, validation_data=val_ds_2,
                                  epochs=8, verbose=0)

# Modelo INCORRECTO: BN recalcula estadisticos
model_bn_wrong = build_bn_demo_model(training_flag=True, label='wrong')
model_bn_wrong.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                      loss='categorical_crossentropy', metrics=['accuracy'])

print("Entrenando modelo INCORRECTO (training=True en BN)...")
h_wrong = model_bn_wrong.fit(train_ds_2, validation_data=val_ds_2,
                              epochs=8, verbose=0)

In [ ]:
# Comparar resultados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep = range(1, 9)
axes[0].plot(ep, h_correct.history['val_accuracy'], 'o-', label='training=False (CORRECTO)',
            color='#4CAF50', linewidth=2)
axes[0].plot(ep, h_wrong.history['val_accuracy'], 's-', label='training=True (INCORRECTO)',
            color='#F44336', linewidth=2)
axes[0].set_title('Val Accuracy — Efecto de training flag en BN', fontweight='bold')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, h_correct.history['val_loss'], 'o-', label='training=False (CORRECTO)',
            color='#4CAF50', linewidth=2)
axes[1].plot(ep, h_wrong.history['val_loss'], 's-', label='training=True (INCORRECTO)',
            color='#F44336', linewidth=2)
axes[1].set_title('Val Loss — Efecto de training flag en BN', fontweight='bold')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Error silencioso: BatchNormalization con training=True\n'
            'Las BN recalculan estadisticos con batches pequenos y desestabilizan el modelo',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nResultados:")
print(f"  training=False (correcto): {max(h_correct.history['val_accuracy']):.4f}")
print(f"  training=True  (incorrecto): {max(h_wrong.history['val_accuracy']):.4f}")

### ✅ Conclusión Ejemplo 2

**Hallazgos del experimento de profundidad:**
- Descongelar 0 capas (FE puro) da buen baseline
- Descongelar ~20-50 capas suele dar el mejor resultado para datasets medianos
- Descongelar TODO puede causar overfitting con pocos datos

**Sobre BatchNormalization:**
- `training=False` es **OBLIGATORIO** durante Fine-Tuning
- Con `training=True`, las BN recalculan media/varianza con batches pequeños → inestabilidad
- Este error es **silencioso**: el modelo entrena sin errores pero con peor rendimiento

---
# 🔴 Ejemplo 3 — Avanzado: Comparación completa + Mixup + TFLite

**Objetivo:** Comparar las 3 estrategias (desde cero, FE, Fine-Tuning) en un problema multiclase real. Aplicar Mixup como regularización avanzada. Exportar a TFLite.

**Dataset:** Oxford-IIIT Pet — 37 razas de gatos y perros (~7,400 imágenes). Un problema realista y desafiante.

### 3.1. Cargar Oxford Pets

In [ ]:
# Cargar Oxford-IIIT Pet (37 razas)
(train_raw_3, val_raw_3), info_3 = tfds.load(
    'oxford_iiit_pet',
    split=['train[:85%]', 'train[85%:]'],
    as_supervised=True, with_info=True
)

NUM_CLASSES_3 = info_3.features['label'].num_classes
CLASS_NAMES_3 = info_3.features['label'].names
print(f"Clases: {NUM_CLASSES_3}")
print(f"Train: {tf.data.experimental.cardinality(train_raw_3).numpy():,}")
print(f"Val:   {tf.data.experimental.cardinality(val_raw_3).numpy():,}")
print(f"\nEjemplos de razas: {CLASS_NAMES_3[:10]}")

In [ ]:
# Visualizar muestras
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, (img, label) in enumerate(train_raw_3.shuffle(1000).take(18)):
    ax = axes[i // 6, i % 6]
    ax.imshow(img.numpy())
    name = CLASS_NAMES_3[label.numpy()]
    ax.set_title(name.replace('_', ' '), fontsize=9)
    ax.axis('off')

plt.suptitle('Oxford-IIIT Pet — 37 razas de gatos y perros', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Preparar datasets
aug_3 = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.15),
])

def prep_train_3(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = aug_3(image, training=True)
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    return image, tf.one_hot(label, NUM_CLASSES_3)

def prep_val_3(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.efficientnet.preprocess_input(tf.cast(image, tf.float32))
    return image, tf.one_hot(label, NUM_CLASSES_3)

train_ds_3 = (train_raw_3.map(prep_train_3, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(2000, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_ds_3 = (val_raw_3.map(prep_val_3, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print("Datasets listos (37 clases)")

### 3.2. Estrategia A: CNN desde cero (baseline)

In [ ]:
# CNN desde cero (como en la Sesion 08)
model_scratch = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES_3, activation='softmax')
], name='cnn_desde_cero')

model_scratch.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print(f"CNN desde cero: {model_scratch.count_params():,} params (todos entrenables)")
print("Entrenando CNN desde cero (esto toma mas tiempo)...")

t0 = time.time()
h_scratch = model_scratch.fit(train_ds_3, validation_data=val_ds_3, epochs=25, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7,
                                                restore_best_weights=True)])
time_scratch = time.time() - t0

scratch_acc = max(h_scratch.history['val_accuracy'])
print(f"\nCNN desde cero: val_acc = {scratch_acc:.4f} | tiempo = {time_scratch:.0f}s")

### 3.3. Estrategia B: Feature Extraction

In [ ]:
# Feature Extraction con EfficientNetB0
base_fe = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_fe.trainable = False

model_fe = models.Sequential([
    base_fe,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES_3, activation='softmax')
], name='feature_extraction')

model_fe.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                loss='categorical_crossentropy', metrics=['accuracy'])

trainable_fe = sum(tf.size(w).numpy() for w in model_fe.trainable_weights)
print(f"Feature Extraction: {trainable_fe:,} params entrenables")
print("Entrenando Feature Extraction...")

t0 = time.time()
h_fe = model_fe.fit(train_ds_3, validation_data=val_ds_3, epochs=15, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                                restore_best_weights=True)])
time_fe = time.time() - t0

fe_acc_3 = max(h_fe.history['val_accuracy'])
print(f"\nFeature Extraction: val_acc = {fe_acc_3:.4f} | tiempo = {time_fe:.0f}s")

### 3.4. Estrategia C: Fine-Tuning (2 fases)

In [ ]:
# Fine-Tuning con EfficientNetB0
base_ft = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_ft.trainable = False

inp_ft = layers.Input(shape=(*IMG_SIZE, 3))
x = base_ft(inp_ft, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
out_ft = layers.Dense(NUM_CLASSES_3, activation='softmax')(x)
model_ft = models.Model(inp_ft, out_ft, name='fine_tuning')

# Fase 1
model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                loss='categorical_crossentropy', metrics=['accuracy'])

print("Fase 1: Feature Extraction (warm-up)...")
t0 = time.time()
h_ft_1 = model_ft.fit(train_ds_3, validation_data=val_ds_3, epochs=8, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                                restore_best_weights=True)])

# Fase 2
base_ft.trainable = True
for layer in base_ft.layers[:-40]:
    layer.trainable = False

model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                loss='categorical_crossentropy', metrics=['accuracy'])

trainable_ft = sum(tf.size(w).numpy() for w in model_ft.trainable_weights)
print(f"Fase 2: Fine-Tuning ({trainable_ft:,} params entrenables)...")

h_ft_2 = model_ft.fit(train_ds_3, validation_data=val_ds_3, epochs=20, verbose=0,
    initial_epoch=h_ft_1.epoch[-1]+1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7,
                                          restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=3, min_lr=1e-7),
    ])
time_ft = time.time() - t0

ft_acc_3 = max(h_ft_2.history['val_accuracy'])
print(f"\nFine-Tuning: val_acc = {ft_acc_3:.4f} | tiempo = {time_ft:.0f}s")

### 3.5. Tabla y gráfico comparativo de las 3 estrategias

In [ ]:
# Tabla comparativa
comparison = pd.DataFrame([
    {'Estrategia': 'CNN desde cero', 'Val Accuracy': f'{scratch_acc:.4f}',
     'Tiempo (s)': f'{time_scratch:.0f}', 'Params entrenables': f'{model_scratch.count_params():,}'},
    {'Estrategia': 'Feature Extraction', 'Val Accuracy': f'{fe_acc_3:.4f}',
     'Tiempo (s)': f'{time_fe:.0f}', 'Params entrenables': f'{trainable_fe:,}'},
    {'Estrategia': 'Fine-Tuning', 'Val Accuracy': f'{ft_acc_3:.4f}',
     'Tiempo (s)': f'{time_ft:.0f}', 'Params entrenables': f'{trainable_ft:,}'},
])

print("\n" + "=" * 75)
print("COMPARACION: 3 ESTRATEGIAS EN OXFORD PETS (37 razas)")
print("=" * 75)
print(comparison.to_string(index=False))
print("=" * 75)

In [ ]:
# Grafico comparativo
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

strats = ['CNN desde\ncero', 'Feature\nExtraction', 'Fine-\nTuning']
accs_all = [scratch_acc, fe_acc_3, ft_acc_3]
times_all = [time_scratch, time_fe, time_ft]
colors_3 = ['#E74C3C', '#3498DB', '#2ECC71']

# 1. Accuracy
bars_acc = axes[0].bar(strats, accs_all, color=colors_3, edgecolor='gray', alpha=0.85)
axes[0].set_title('Val Accuracy', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, max(accs_all) + 0.08])
for bar, acc in zip(bars_acc, accs_all):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{acc:.3f}', ha='center', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

# 2. Tiempo
bars_time = axes[1].bar(strats, times_all, color=colors_3, edgecolor='gray', alpha=0.85)
axes[1].set_title('Tiempo de Entrenamiento', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Segundos')
for bar, t in zip(bars_time, times_all):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{t:.0f}s', ha='center', fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Curvas de val accuracy superpuestas
ep_s = range(1, len(h_scratch.history['val_accuracy'])+1)
axes[2].plot(ep_s, h_scratch.history['val_accuracy'], '-', label='CNN desde cero',
            color=colors_3[0], linewidth=2)

ep_fe = range(1, len(h_fe.history['val_accuracy'])+1)
axes[2].plot(ep_fe, h_fe.history['val_accuracy'], '-', label='Feature Extraction',
            color=colors_3[1], linewidth=2)

ep_ft1 = list(range(1, len(h_ft_1.history['val_accuracy'])+1))
ep_ft2 = list(range(ep_ft1[-1]+1, ep_ft1[-1]+1+len(h_ft_2.history['val_accuracy'])))
ft_all_acc = h_ft_1.history['val_accuracy'] + h_ft_2.history['val_accuracy']
axes[2].plot(ep_ft1 + ep_ft2, ft_all_acc, '-', label='Fine-Tuning',
            color=colors_3[2], linewidth=2)
axes[2].axvline(x=ep_ft1[-1]+0.5, color='gray', linestyle=':', alpha=0.5)

axes[2].set_title('Val Accuracy por Epoca', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Epoca'); axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=10); axes[2].grid(True, alpha=0.3)

plt.suptitle('Comparacion: CNN desde cero vs Feature Extraction vs Fine-Tuning\n'
            'Dataset: Oxford-IIIT Pet (37 razas)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6. Confusion Matrix del mejor modelo (Fine-Tuning)

In [ ]:
# Confusion matrix del modelo Fine-Tuned
y_true_3, y_pred_3 = [], []
for images, labels in val_ds_3:
    preds = model_ft.predict(images, verbose=0)
    y_pred_3.extend(np.argmax(preds, axis=1))
    y_true_3.extend(np.argmax(labels.numpy(), axis=1))

y_true_3 = np.array(y_true_3)
y_pred_3 = np.array(y_pred_3)

# Para 37 clases, la confusion matrix completa es muy grande
# Mostramos las top-10 confusiones
cm_3 = confusion_matrix(y_true_3, y_pred_3)

# Encontrar pares mas confundidos
confusions = []
for i in range(NUM_CLASSES_3):
    for j in range(NUM_CLASSES_3):
        if i != j and cm_3[i, j] > 0:
            confusions.append((CLASS_NAMES_3[i], CLASS_NAMES_3[j], cm_3[i, j]))

confusions.sort(key=lambda x: x[2], reverse=True)

print("Top-10 confusiones (Fine-Tuning en Oxford Pets)")
print("-" * 55)
for real, pred, count in confusions[:10]:
    print(f"  {real:25s} -> {pred:25s} : {count} errores")

In [ ]:
# Visualizar las confusiones mas comunes
fig, ax = plt.subplots(figsize=(10, 8))

# Obtener las 15 clases mas confundidas
top_confused = set()
for real, pred, _ in confusions[:15]:
    top_confused.add(CLASS_NAMES_3.index(real) if real in CLASS_NAMES_3 else 0)
    top_confused.add(CLASS_NAMES_3.index(pred) if pred in CLASS_NAMES_3 else 0)
top_confused = sorted(list(top_confused))[:12]

cm_subset = cm_3[np.ix_(top_confused, top_confused)]
labels_subset = [CLASS_NAMES_3[i].replace('_', ' ') for i in top_confused]

sns.heatmap(cm_subset, annot=True, fmt='d', cmap='YlOrRd',
           xticklabels=labels_subset, yticklabels=labels_subset, ax=ax,
           linewidths=0.5, linecolor='white')

ax.set_xlabel('Prediccion', fontsize=12, fontweight='bold')
ax.set_ylabel('Real', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix (razas mas confundidas) — Fine-Tuning',
            fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.7. Mixup: Regularización avanzada para Fine-Tuning

**Mixup** crea imágenes sintéticas mezclando linealmente pares de imágenes y sus etiquetas. Es una forma poderosa de regularización, especialmente útil cuando tenemos pocas imágenes.

In [ ]:
# Implementar Mixup
def mixup_batch(images, labels, alpha=0.2):
    '''Mezcla lineal de pares de imagenes y etiquetas.'''
    batch_size = tf.shape(images)[0]
    # Coeficiente de mezcla (distribucion Beta)
    lam = tf.random.uniform([], 0, alpha)
    # Permutar indices
    indices = tf.random.shuffle(tf.range(batch_size))
    # Mezclar
    mixed_images = lam * images + (1 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

# Visualizar Mixup
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

sample_batch = next(iter(train_ds_3))
imgs_sample, labels_sample = sample_batch

# Originales en fila 1
for i in range(5):
    axes[0, i].imshow(((imgs_sample[i].numpy() + 1) / 2).clip(0, 1))
    class_idx = np.argmax(labels_sample[i].numpy())
    axes[0, i].set_title(f'Original\n{CLASS_NAMES_3[class_idx][:12]}', fontsize=9)
    axes[0, i].axis('off')

# Mixup en fila 2
mixed_imgs, mixed_labels = mixup_batch(imgs_sample, labels_sample, alpha=0.3)
for i in range(5):
    axes[1, i].imshow(((mixed_imgs[i].numpy() + 1) / 2).clip(0, 1))
    top2 = np.argsort(mixed_labels[i].numpy())[-2:][::-1]
    axes[1, i].set_title(f'Mixup\n{CLASS_NAMES_3[top2[0]][:8]}+{CLASS_NAMES_3[top2[1]][:8]}',
                         fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Mixup', fontsize=12, fontweight='bold')
plt.suptitle('Mixup: Mezcla lineal de pares de imagenes y etiquetas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Entrenar Fine-Tuning CON Mixup
base_mx = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_mx.trainable = False

inp_mx = layers.Input(shape=(*IMG_SIZE, 3))
x = base_mx(inp_mx, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
out_mx = layers.Dense(NUM_CLASSES_3, activation='softmax')(x)
model_mx = models.Model(inp_mx, out_mx, name='fine_tuning_mixup')

# Fase 1 (sin mixup)
model_mx.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                loss='categorical_crossentropy', metrics=['accuracy'])
h_mx_1 = model_mx.fit(train_ds_3, validation_data=val_ds_3, epochs=5, verbose=0)

# Fase 2 (con mixup)
base_mx.trainable = True
for layer in base_mx.layers[:-40]:
    layer.trainable = False

model_mx.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                loss='categorical_crossentropy', metrics=['accuracy'])

# Dataset con Mixup
train_ds_mixup = train_ds_3.map(
    lambda x, y: mixup_batch(x, y, alpha=0.2),
    num_parallel_calls=tf.data.AUTOTUNE)

print("Entrenando Fine-Tuning + Mixup...")
h_mx_2 = model_mx.fit(train_ds_mixup, validation_data=val_ds_3, epochs=20, verbose=0,
    initial_epoch=h_mx_1.epoch[-1]+1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7,
                                          restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=3, min_lr=1e-7),
    ])

mx_acc = max(h_mx_2.history['val_accuracy'])
print(f"\nFine-Tuning + Mixup: val_acc = {mx_acc:.4f}")
print(f"Fine-Tuning sin Mixup: val_acc = {ft_acc_3:.4f}")
print(f"Diferencia: {(mx_acc-ft_acc_3)*100:+.2f}pp")

### 3.8. Exportar a TFLite para despliegue móvil

In [ ]:
import os

# Guardar modelo keras
model_ft.save('modelo_finetuned_pets.keras')
print(f"Modelo Keras guardado: {os.path.getsize('modelo_finetuned_pets.keras')/1024/1024:.1f} MB")

# Convertir a TFLite con cuantizacion float16
converter = tf.lite.TFLiteConverter.from_keras_model(model_ft)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
with open('modelo_finetuned_pets.tflite', 'wb') as f:
    f.write(tflite_model)

keras_size = os.path.getsize('modelo_finetuned_pets.keras') / 1024 / 1024
tflite_size = len(tflite_model) / 1024 / 1024
reduction = (1 - tflite_size / keras_size) * 100

print(f"\nComparacion de tamanos:")
print(f"  Keras:  {keras_size:.1f} MB")
print(f"  TFLite: {tflite_size:.1f} MB")
print(f"  Reduccion: {reduction:.0f}%")

In [ ]:
# Verificar TFLite con una prediccion
interpreter = tf.lite.Interpreter(model_path='modelo_finetuned_pets.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Info:")
print(f"  Input shape: {input_details[0]['shape']}")
print(f"  Input dtype: {input_details[0]['dtype']}")
print(f"  Output shape: {output_details[0]['shape']}")

# Predecir con TFLite
sample_img, sample_label = next(iter(val_raw_3))
img_processed = tf.image.resize(tf.cast(sample_img, tf.float32), IMG_SIZE)
img_processed = tf.keras.applications.efficientnet.preprocess_input(img_processed)
img_processed = tf.expand_dims(img_processed, 0)

interpreter.set_tensor(input_details[0]['index'], img_processed.numpy().astype(np.float32))
interpreter.invoke()
tflite_pred = interpreter.get_tensor(output_details[0]['index'])

pred_class = CLASS_NAMES_3[np.argmax(tflite_pred)]
true_class = CLASS_NAMES_3[sample_label.numpy()]
confidence = np.max(tflite_pred)

print(f"\nPrediccion TFLite:")
print(f"  Real: {true_class}")
print(f"  Pred: {pred_class} ({confidence:.1%})")
print(f"  Correcto: {'Si' if pred_class == true_class else 'No'}")

### ✅ Conclusión Ejemplo 3

| Estrategia | Val Accuracy | Tiempo | Params entrenables |
|------------|-------------|--------|-------------------|
| **CNN desde cero** | ~30-50% | Largo | Todos (~1M) |
| **Feature Extraction** | ~75-85% | Corto | ~10K |
| **Fine-Tuning** | ~85-92% | Medio | ~800K |
| **Fine-Tuning + Mixup** | ~87-93% | Medio | ~800K |

**Observaciones:**
- Con 37 clases y ~200 imgs/clase, CNN desde cero tiene un rendimiento muy pobre
- Feature Extraction ya logra resultados decentes sin tocar el backbone
- Fine-Tuning mejora significativamente al adaptar features
- Mixup agrega regularización extra, útil cuando hay riesgo de overfitting
- TFLite reduce el modelo ~50% para despliegue móvil

---
# 📋 Resumen de la Sesión 09-B

## Checklist de Fine-Tuning Profesional

| # | Paso | Verificado |
|---|------|-----------|
| 1 | Feature Extraction como warm-up (Fase 1) | |
| 2 | Descongelar las últimas N capas (bloques completos) | |
| 3 | LR de Fine-Tuning = 10x-100x menor que Fase 1 | |
| 4 | `training=False` en BN del backbone | |
| 5 | Recompilar el modelo después de cambiar `trainable` | |
| 6 | `initial_epoch` para continuar numeración | |
| 7 | Callbacks: EarlyStopping + ReduceLROnPlateau + ModelCheckpoint | |
| 8 | Data Augmentation para regularizar | |
| 9 | Comparar contra baselines (desde cero + FE) | |
| 10 | Exportar modelo final (.keras + .tflite) | |

## Resumen de las 3 estrategias

```
                    Accuracy
                    ▲
  Fine-Tuning  ●    │  ████████████████████████████ 85-98%
  Feature Ext  ●    │  ██████████████████████       75-95%
  Desde cero   ●    │  ███████████████              50-90%
                    └──────────────────────────────→ Datos necesarios
```

## Conexión con su tarea de investigación
Si ya tienen su dataset propio de la Sesión 08, ahora pueden:
1. Aplicar Feature Extraction como nueva baseline
2. Aplicar Fine-Tuning para su mejor resultado
3. Comparar las 3 estrategias en su informe

---
*IS056B — Ingeniería del Conocimiento | UNCP 2026-II | Msc. Jaime Huaytalla Pariona*